# The Polyglot's Shorthand v1.2 - 01 - Teacher diagnostic (v1.1 train split only)

Fine-tunes hing-roberta, hing-roberta-mixed, hing-bert and MuRIL (3 learning rates each) on the v1.1 TRAIN split, early-stops on validation NLL and saves logits for train/val/test/test_stress/audit/contrast.

**Expected GPU time:** ~35-60 min on a T4 (12 short runs; most time is model download and checkpoint writes). Hard guard: 105 min.

**Colab (recommended):** Runtime > Change runtime type > **T4 GPU**. Put `bundle.zip` at `MyDrive/polyglot12/bundle.zip`
(or upload it when prompted). Then **Runtime > Run all**. Checkpoints go to `MyDrive/polyglot12/01_teacher/`; after a
disconnect simply **Run all** again - finished runs are skipped and the interrupted run resumes from its last epoch.

**Kaggle:** Settings > Accelerator **GPU T4 x2** (not P100) and **Internet ON**. Add `bundle.zip` as a dataset
(Add Data > Upload). Run all. Outputs are in `/kaggle/working` (only kept across sessions if you use Save Version).

At the end `outputs.zip` is downloaded (Colab) or appears in the Output panel (Kaggle).


In [ ]:
# ---- 1. Platform, persistent storage, bundle location --------------------------------
import glob, json, os, shutil, subprocess, sys, time, zipfile
NOTEBOOK = '01_teacher'
IS_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
IS_KAGGLE = os.path.exists('/kaggle/working')
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')                      # checkpoints survive disconnects here
    PERSIST = f'/content/drive/MyDrive/polyglot12/{NOTEBOOK}'
    WORK = '/content/v12bundle'
    OUT_ZIP = f'/content/drive/MyDrive/polyglot12/{NOTEBOOK}_outputs.zip'
elif IS_KAGGLE:
    PERSIST = f'/kaggle/working/{NOTEBOOK}'
    WORK = '/kaggle/tmp/v12bundle'
    OUT_ZIP = '/kaggle/working/outputs.zip'
else:
    raise SystemExit('Run this notebook on Google Colab or Kaggle (GPU).')
os.makedirs(PERSIST, exist_ok=True)
print('platform:', 'colab' if IS_COLAB else 'kaggle', '| persistent dir:', PERSIST)


In [ ]:
# ---- 2. GPU check + dependencies (torch is preinstalled; do not reinstall it) -----------
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2.'
cap = torch.cuda.get_device_capability(0)
print('GPU:', torch.cuda.get_device_name(0), 'compute capability', cap)
if cap < (7, 0):
    print('WARNING: P100-class GPU. Recent PyTorch wheels may not support it; on Kaggle pick "GPU T4 x2".')
if IS_KAGGLE:
    import urllib.request
    try:
        urllib.request.urlopen('https://huggingface.co', timeout=10)
    except Exception as e:
        raise SystemExit('No internet: Kaggle > Settings > Internet ON (needed to download models).') from e
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.46,<6', 'sentencepiece', 'protobuf'], check=True)


In [ ]:
# ---- 3. Unpack bundle.zip and run the hash-consistency test ----------------------------
def find_bundle():
    cands = (['/content/drive/MyDrive/polyglot12/bundle.zip', '/content/bundle.zip'] if IS_COLAB
             else glob.glob('/kaggle/input/**/bundle.zip', recursive=True) + ['/kaggle/working/bundle.zip'])
    for c in cands:
        if os.path.exists(c):
            return ('zip', c)
    if IS_KAGGLE:  # Kaggle datasets auto-extract zips: look for the unpacked tree instead
        hits = glob.glob('/kaggle/input/**/polyglot12/teacher.py', recursive=True)
        if hits:
            return ('dir', str(os.path.dirname(os.path.dirname(hits[0]))))
    if IS_COLAB:
        from google.colab import files
        print('bundle.zip not found on Drive (MyDrive/polyglot12/bundle.zip) - upload it now:')
        up = files.upload()
        return ('zip', '/content/' + next(iter(up)))
    raise SystemExit('bundle.zip not found. Kaggle: Add Data > upload bundle.zip as a dataset.')

kind, src = find_bundle()
shutil.rmtree(WORK, ignore_errors=True)
if kind == 'zip':
    with zipfile.ZipFile(src) as z:
        z.extractall(WORK)
else:
    shutil.copytree(src, WORK)
print('bundle from', src)
sys.path.insert(0, WORK)
r = subprocess.run([sys.executable, 'tests/test_hashing.py'], cwd=WORK, capture_output=True, text=True)
print(r.stdout, r.stderr)
open(f'{PERSIST}/hash_test.txt', 'w').write(r.stdout + r.stderr)
assert r.returncode == 0, 'Hashing differs from the Mac fixtures - stop and report this.'


In [ ]:
# ---- 4. Train (auto-resumes: just "Run all" again after a disconnect) -------------------
cmd = [sys.executable, '-u', '-m', 'polyglot12.teacher', '--config', 'configs/teacher_v11data.json', '--out', PERSIST]
print(' '.join(cmd))
t0 = time.time()
p = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                     env={**os.environ, 'TOKENIZERS_PARALLELISM': 'false', 'HF_HUB_DISABLE_PROGRESS_BARS': '1'})
with open(f'{PERSIST}/train_log.txt', 'a') as log:
    for line in p.stdout:
        print(line, end='')
        log.write(line)
rc = p.wait()
print(f'finished with exit code {rc} after {(time.time() - t0) / 60:.1f} min')
assert rc == 0, 'Training failed - see the log above (re-running resumes from the last checkpoint).'


In [ ]:
# ---- 5. Package outputs.zip (logits, status, summary, logs; no weights) ------------------
import importlib.metadata as md
env = {'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
       'transformers': md.version('transformers'), 'python': sys.version.split()[0],
       'platform': 'colab' if IS_COLAB else 'kaggle'}
json.dump(env, open(f'{PERSIST}/env.json', 'w'), indent=1)
with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(PERSIST):
        for f in fs:
            full = os.path.join(root, f)
            if f.endswith(('.json', '.npz', '.txt')) or ('no' == 'yes' and f == 'student.pt'):
                z.write(full, os.path.relpath(full, PERSIST))
print('wrote', OUT_ZIP, f'{os.path.getsize(OUT_ZIP) / 1e6:.1f} MB')
if IS_COLAB:
    from google.colab import files
    files.download(OUT_ZIP)
else:
    print('Kaggle: download outputs.zip from the Output panel (right side, /kaggle/working).')
